# Delaware PUMS — Wealth-Calibrated Chile–Barbados Measurement

Extends the income-level calibration (`delaware_calibration.ipynb`) to housing wealth using the household-level PUMS file (`psam_h10.csv`).  
Produces the first-pass nucleation deficit measurement on real Delaware microdata.

**Steps**
1. Load & inspect household data
2. Build wealth proxy and equivalised metrics
3. Cumulant decomposition on wealth (Black vs White)
4. Sheaf gluing obstruction check on wealth
5. ICR proxy using mortgage / owner-cost data
6. Chile–Barbados interval statement — nucleation deficit

In [1]:
import pathlib
import sys
import logging
import numpy as np
import pandas as pd
import torch
import scipy.stats as stats

# Locate project root
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"Project root not found from {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from structural_impedance.cumulant import (
    admit_per_component_standardized,
    cumulant_difference,
)
from structural_impedance.sheaf_gluing import (
    cocycle_disagreement,
    sheaf_status_and_kappa,
)

HH_PATH = ROOT / "data" / "pums" / "psam_h10.csv"
P_PATH  = ROOT / "data" / "pums" / "psam_p10.csv"
assert HH_PATH.exists(), f"Household file not found: {HH_PATH}"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("delaware_wealth")

print(f"ROOT    : {ROOT}")
print(f"HH_PATH : {HH_PATH}")

ROOT    : /Users/alanevans/OTS_3.0_1160
HH_PATH : /Users/alanevans/OTS_3.0_1160/data/pums/psam_h10.csv


---
## Step 1 — Load & Inspect Household Data

`TYPEHUGQ == 1` selects household records; types 2 and 3 are institutional and non-institutional group quarters (dorms, barracks, shelters) that lack housing and income variables.

In [2]:
# Cell 1.1 — Load and filter to household records
hh_raw = pd.read_csv(HH_PATH, low_memory=False)
hh = hh_raw[hh_raw["TYPEHUGQ"] == 1].copy()

print(f"Raw rows       : {len(hh_raw):,}")
print(f"Household recs : {len(hh):,}  (TYPEHUGQ==1)")
print()

KEY = ["SERIALNO", "TEN", "VALP", "HINCP", "NP", "MRGP", "SMOCP", "RNTP", "GRNTP", "HHLDRRAC1P"]
print("=== dtypes ===")
print(hh[KEY].dtypes)
print()
print("=== TEN value counts ===")
print(hh["TEN"].value_counts(dropna=False))
print()
print("=== HHLDRRAC1P value counts ===")
print(hh["HHLDRRAC1P"].value_counts(dropna=False))
print()
print("=== key wealth variable non-null counts ===")
print(hh[["VALP","HINCP","MRGP","SMOCP","RNTP","GRNTP"]].notna().sum())

Raw rows       : 5,082
Household recs : 4,765  (TYPEHUGQ==1)

=== dtypes ===
SERIALNO       object
TEN           float64
VALP          float64
HINCP         float64
NP              int64
MRGP          float64
SMOCP         float64
RNTP          float64
GRNTP         float64
HHLDRRAC1P    float64
dtype: object

=== TEN value counts ===
1.0    2035
2.0    1415
3.0     701
NaN     556
4.0      58
Name: TEN, dtype: int64

=== HHLDRRAC1P value counts ===
1.0    3190
NaN     556
2.0     527
9.0     216
6.0     152
8.0      96
3.0      16
5.0       8
7.0       3
4.0       1
Name: HHLDRRAC1P, dtype: int64

=== key wealth variable non-null counts ===
VALP     3488
HINCP    4209
MRGP     1948
SMOCP    3450
RNTP      756
GRNTP     701
dtype: int64


In [3]:
# Cell 1.2 — SERIALNO cross-check with person file
# (informational only; HHLDRRAC1P provides race directly so no join is needed)
df_person = pd.read_csv(P_PATH, usecols=["SERIALNO"])
hh_ids = set(hh["SERIALNO"].astype(str))
p_ids  = set(df_person["SERIALNO"].astype(str))
overlap = len(hh_ids & p_ids)
print(f"HH records              : {len(hh_ids):,}")
print(f"Person-file SERIALNO    : {len(p_ids):,}")
print(f"Overlap (linked)        : {overlap:,}")
print(f"HH only (GQ spillover)  : {len(hh_ids - p_ids):,}")

HH records              : 4,765
Person-file SERIALNO    : 4,526
Overlap (linked)        : 4,209
HH only (GQ spillover)  : 556


---
## Step 2 — Race Recode, Wealth Proxy, and Equivalised Metrics

**Race source:** `HHLDRRAC1P` (householder race, household-level field) — same coding as person-level `RAC1P`. No person-file join needed.

**Wealth proxy (gross housing value):**
- Owners (`TEN` ∈ {1, 2}): `gross_home_value = VALP × ADJHSG_mult` (self-reported property value, inflation-adjusted)
- Renters (`TEN` ∈ {3, 4}): `gross_home_value = 0` (lower bound; non-housing assets absent from PUMS)
- Fallback: missing owner VALP → race-group median; logged.

**Per-adult housing wealth:** `gross_home_value / NP` (stock ÷ household size).
**Income (flow):** `equiv_income = HINCP_adj / sqrt(NP)` (OECD modified scale).

**Adjustment factors:** `ADJHSG` scales housing values; `ADJINC` scales income. Both are vintage-level constants (same value for every row in this file).

In [4]:
# Cell 2.1 — Race recode from HHLDRRAC1P
race_map = {1.0: "White", 2.0: "Black"}
hh["hh_race"] = hh["HHLDRRAC1P"].map(race_map).fillna("Other")

print("=== hh_race value counts ===")
print(hh["hh_race"].value_counts())

=== hh_race value counts ===
White    3190
Other    1048
Black     527
Name: hh_race, dtype: int64


In [5]:
# Cell 2.2 — Gross home value proxy with adjustment factors and logged imputation
# NOTE: this is GROSS home value (VALP), not net worth. See the wealth-proxy caveat below.

# ADJHSG and ADJINC are vintage-level multipliers (same for every row)
# Divide by 1_000_000 to get the actual multiplier (e.g. 1000000 -> 1.0, 1015250 -> 1.01525)
adjhsg = float(hh["ADJHSG"].iloc[0]) / 1_000_000
adjinc = float(hh["ADJINC"].iloc[0]) / 1_000_000
print(f"ADJHSG multiplier : {adjhsg:.6f}")
print(f"ADJINC multiplier : {adjinc:.6f}")

# Adjusted housing value (NaN for non-owners / missing)
hh["VALP_adj"] = hh["VALP"] * adjhsg

# Compute race-group median VALP for imputation fallback (owners only)
owners_mask = hh["TEN"].isin([1.0, 2.0])
valp_medians = (
    hh[owners_mask & hh["VALP_adj"].notna()]
    .groupby("hh_race")["VALP_adj"]
    .median()
)
print("\nMedian VALP_adj by race (owner fallback):")
print(valp_medians)

# Vectorized gross-home-value assignment
hh["gross_home_value"] = 0.0  # default: renters and GQ

# Owners with valid VALP
valid_valp = owners_mask & hh["VALP_adj"].notna() & (hh["VALP_adj"] > 0)
hh.loc[valid_valp, "gross_home_value"] = hh.loc[valid_valp, "VALP_adj"]

# Owners with missing/zero VALP: impute and log
for race_label in ["White", "Black", "Other"]:
    impute_mask = owners_mask & (hh["hh_race"] == race_label) & ~valid_valp
    n_imp = int(impute_mask.sum())
    if n_imp > 0:
        fallback = valp_medians.get(race_label, float(valp_medians.median()))
        log.warning("VALP imputed: %d %s-race owner records -> median $%.0f",
                    n_imp, race_label, fallback)
        hh.loc[impute_mask, "gross_home_value"] = fallback

print(f"\ngross_home_value > 0 : {(hh['gross_home_value'] > 0).sum():,} households")
print(f"gross_home_value = 0 : {(hh['gross_home_value'] == 0).sum():,} households (renters + missing)")

ADJHSG multiplier : 1.000000
ADJINC multiplier : 1.015250

Median VALP_adj by race (owner fallback):
hh_race
Black    330000.0
Other    365000.0
White    400000.0
Name: VALP_adj, dtype: float64

gross_home_value > 0 : 3,450 households
gross_home_value = 0 : 1,315 households (renters + missing)


In [6]:
# Cell 2.3 — Per-adult housing wealth, equivalised income, nucleation filter
hh["HINCP_adj"] = hh["HINCP"] * adjinc
# Housing wealth is a STOCK: divide by household size (NP), not the sqrt(NP) income scale.
hh["home_value_per_adult"] = hh["gross_home_value"] / hh["NP"].clip(lower=1)
# Income is a FLOW: OECD sqrt(NP) equivalisation is standard and retained.
hh["equiv_income"] = hh["HINCP_adj"] / np.sqrt(hh["NP"].clip(lower=1))

# Nucleation threshold: HINCP > 0 and equiv_income > $30,000
hh_above = hh[(hh["HINCP_adj"] > 0) & (hh["equiv_income"] > 30_000)].copy()

print("=== Households above nucleation threshold ($30k equiv income) ===")
print(hh_above["hh_race"].value_counts())
print()
print(hh_above.groupby("hh_race")[["equiv_income", "home_value_per_adult"]].median())

=== Households above nucleation threshold ($30k equiv income) ===
White    2660
Black     389
Other     378
Name: hh_race, dtype: int64

         equiv_income  home_value_per_adult
hh_race                                    
Black    61451.397662         110000.000000
Other    73562.435030         122833.333333
White    75574.288521         175000.000000


> **Important — wealth proxy definition.** The wealth proxy is **gross home value** (`VALP`, vintage-adjusted), stored as `gross_home_value`. It is **not net worth**: mortgage debt is not subtracted, and non-housing assets (stocks, retirement, savings) are absent. Renters are assigned 0. Per-adult housing wealth is `gross_home_value / NP` — a **stock** divided by household size, *not* the √NP income-equivalisation scale. All dollar figures are a **lower-bound housing-wealth component**, not total net worth. The Chile–Barbados \$200k–\$300k band is a net-worth expectation and is reported for reference only; comparisons are stated as housing-wealth gaps and percentile positions, not net-worth deficits. The median tables are robust to this definition.

---
## Step 3 — Cumulant Divergence on Wealth

`cumulant_difference(X, Y, k)` compares the **separate marginal cumulants** of two independent populations (Black vs White per-adult housing wealth) and returns `‖κ_k(X) − κ_k(Y)‖`. It does **not** require equal sample sizes and does **not** depend on an arbitrary row-pairing.

The standardized admission gate (`admit_per_component_standardized`) is retained as a **noise guard**, with τ in σ-units. A permutation null control confirms the gate stays silent on noise and that `‖Δκ‖` collapses when there is no real group difference.

Conformance: axioms §0.5, §0.5.1; Sturmfels & Zwiernik arXiv:1011.1722.

In [7]:
# Cell 3.1 — Distributional cumulant DIFFERENCE on home_value_per_adult
# cumulant_difference compares the SEPARATE marginal cumulants of two independent
# populations (Black vs White); it does NOT require equal sample sizes.
black_w = hh_above[hh_above["hh_race"] == "Black"]["home_value_per_adult"].values
white_w = hh_above[hh_above["hh_race"] == "White"]["home_value_per_adult"].values

if len(black_w) == 0 or len(white_w) == 0:
    raise RuntimeError("One or both wealth subgroups are empty above nucleation threshold.")

Xf = torch.tensor(black_w, dtype=torch.float64).unsqueeze(1)
Yf = torch.tensor(white_w, dtype=torch.float64).unsqueeze(1)

d3 = float(cumulant_difference(Xf, Yf, k_order=3))
d4 = float(cumulant_difference(Xf, Yf, k_order=4))

# Equalized pair for the standardized admission gate (cross-block needs equal n).
n = min(len(black_w), len(white_w))
rng = np.random.default_rng(42)
X = torch.tensor(rng.choice(black_w, n, replace=False), dtype=torch.float64).unsqueeze(1)
Y = torch.tensor(rng.choice(white_w, n, replace=False), dtype=torch.float64).unsqueeze(1)

print(f"n_black : {len(black_w):,}    n_white : {len(white_w):,}    (no equalization for difference)")
print()
print(f"||Δκ_3|| (Black vs White, 3rd-order marginal cumulant difference) : {d3:.6e}")
print(f"||Δκ_4|| (Black vs White, 4th-order marginal cumulant difference) : {d4:.6e}")

n_black : 389    n_white : 2,660    (no equalization for difference)

||Δκ_3|| (Black vs White, 3rd-order marginal cumulant difference) : 1.167095e+17
||Δκ_4|| (Black vs White, 4th-order marginal cumulant difference) : 2.976507e+23


In [8]:
# Cell 3.2 — Standardized per-component admission gate (axioms §0.5.1)
# Standardizes X, Y to unit variance so τ is in σ-units (default 0.2), not dollar-units.
# This gate is a NOISE GUARD on the cross-block pairing; the divergence signal lives
# in ||Δκ|| (Cell 3.1). The null control (next cell) confirms both behave correctly.
admit, diag = admit_per_component_standardized(X, Y)

print(f"admit              : {admit}")
print(f"blocking_component : {diag.get('blocking_component')}")
print()
print("full diagnostic (standardized, σ-units):")
for k, v in diag.items():
    print(f"  {k:<22}: {v}")

admit              : True
blocking_component : None

full diagnostic (standardized, σ-units):
  ||k3||                : 0.3255859842611446
  ||k4||                : 3.2429840856570284
  tau_3                 : 0.2
  tau_4                 : 0.2
  blocking_component    : None


In [9]:
# Cell 3.3 — PERMUTATION NULL CONTROL (same population, random split)
all_data = torch.cat([X, Y], dim=0)
n_total = all_data.shape[0]
g = torch.Generator().manual_seed(42)
idx = torch.randperm(n_total, generator=g)
split = n_total // 2
null_X = all_data[idx[:split]]
null_Y = all_data[idx[split:2 * split]]

null_admit, null_diag = admit_per_component_standardized(null_X, null_Y)
null_d3 = float(cumulant_difference(null_X, null_Y, k_order=3))
null_d4 = float(cumulant_difference(null_X, null_Y, k_order=4))

print("NULL CONTROL (same population, random split):")
print(f"  gate admit         = {null_admit}   blocking = {null_diag.get('blocking_component')}")
print(f"  null ||k3|| (σ)     = {null_diag.get('||k3||'):.4f}   null ||k4|| (σ) = {null_diag.get('||k4||'):.4f}")
print()
print(f"  ||Δκ_3||  real = {d3:.4e}   null = {null_d3:.4e}   ratio = {d3 / max(null_d3, 1e-9):.1f}x")
print(f"  ||Δκ_4||  real = {d4:.4e}   null = {null_d4:.4e}   ratio = {d4 / max(null_d4, 1e-9):.1f}x")
print()
if null_admit:
    print("  WARNING: gate fires on null — admission may be noise.")
else:
    print("  Gate silent on null (noise guard OK). ||Δκ|| real ≫ null ⇒ genuine divergence.")

NULL CONTROL (same population, random split):
  gate admit         = True   blocking = None
  null ||k3|| (σ)     = 0.2356   null ||k4|| (σ) = 3.6693

  ||Δκ_3||  real = 1.1671e+17   null = 6.4407e+16   ratio = 1.8x
  ||Δκ_4||  real = 2.9765e+23   null = 2.2155e+23   ratio = 1.3x



---
## Step 4 — Sheaf Gluing on Wealth

Local sections: `[mean, var, skew, kurt]` of `home_value_per_adult` for (Black, White, All) above threshold.
Overlaps: `[[0,2],[1,2]]` — Black↔All and White↔All.

**Sheaf normalization note:** The sheaf-gluing obstruction test uses **bootstrap standard-error normalization**. Each summary statistic (mean, variance, skew, kurtosis) is divided by its bootstrap sampling SE computed on the pooled population. Tolerance is in SE units (`tol=8` = 8 standard errors of disagreement); the value used is the larger of 8 SE or 1.5× the null's maximum disagreement, so a **null control** (two random splits of the same population) is valid by construction and any real obstruction exceeds the measured noise ceiling. Note: Delaware has a small Black sample (n≈389), so sampling SE is large and the per-adult divergence may not clear the threshold — that is an honest statement of statistical power, not absence of structure (the national notebook, n≈70k Black, is the powered test).

Conformance: Curry 2014 cellular sheaf cocycle equality (axioms §4).

In [10]:
# Cell 4.1 — Build bootstrap-SE-normalized sections [3, 4]
def stat_vec(arr: np.ndarray) -> np.ndarray:
    return np.array([float(np.mean(arr)), float(np.var(arr)),
                     float(stats.skew(arr)), float(stats.kurtosis(arr))])

def bootstrap_se(pop: np.ndarray, n: int, n_boot: int = 300, seed: int = 0) -> np.ndarray:
    """Sampling SE of each summary statistic at sample size n, on the pooled population."""
    r = np.random.default_rng(seed)
    samples = np.array([stat_vec(r.choice(pop, n, replace=False)) for _ in range(n_boot)])
    return samples.std(axis=0).clip(min=1e-8)

black_w_all = hh_above[hh_above["hh_race"] == "Black"]["home_value_per_adult"].values
white_w_all = hh_above[hh_above["hh_race"] == "White"]["home_value_per_adult"].values
all_w       = hh_above["home_value_per_adult"].values

for label, arr in [("Black", black_w_all), ("White", white_w_all), ("All", all_w)]:
    if len(arr) < 4:
        raise RuntimeError(f"Subgroup '{label}' has only {len(arr)} samples; kurtosis undefined.")

# SE in units of the smaller group's sampling noise (most conservative).
n_se = min(len(black_w_all), len(white_w_all))
se = bootstrap_se(all_w, n_se, seed=12345)

raw_sections = np.array([stat_vec(black_w_all), stat_vec(white_w_all), stat_vec(all_w)])  # [3,4]
normed = raw_sections / se                                    # SE units
sections = torch.tensor(normed, dtype=torch.float64)          # [3, 4]
overlaps = torch.tensor([[0, 2], [1, 2]], dtype=torch.long)   # Black↔All, White↔All

print("Raw sections [mean, var, skew, kurt]:")
for label, row in zip(["Black", "White", "All"], raw_sections):
    print(f"  {label:<6}: {row}")
print()
print(f"Bootstrap SE per statistic (n={n_se:,}): {se}")
print()
print("SE-normalized sections (statistic / its sampling SE):")
for label, row in zip(["Black", "White", "All"], normed):
    print(f"  {label:<6}: {row.tolist()}")

Raw sections [mean, var, skew, kurt]:
  Black : [1.45149425e+05 2.99252716e+10 3.18134802e+00 1.84610486e+01]
  White : [2.34689054e+05 8.67064717e+10 5.21623344e+00 4.17906909e+01]
  All   : [2.17906706e+05 7.67821097e+10 5.19970413e+00 4.34888001e+01]

Bootstrap SE per statistic (n=389): [1.40479789e+04 2.53003834e+10 1.17572079e+00 1.63184736e+01]

SE-normalized sections (statistic / its sampling SE):
  Black : [10.33240620949791, 1.1827991343971778, 2.7058703392432073, 1.1312975117446005]
  White : [16.70625045921591, 3.427081340023632, 4.436625992957198, 2.5609436226612807]
  All   : [15.511605384617017, 3.034820007500384, 4.422567124202919, 2.6650041605149344]


In [11]:
# Cell 4.1b — SHEAF NULL CONTROL + adaptive tolerance calibration
# Two random splits of the pooled population define the noise ceiling. tol is set to
# max(8 SE, 1.5 × null_max) so the null is valid by construction.
rng_s = np.random.default_rng(2024)
perm = rng_s.permutation(len(all_w))
half = len(all_w) // 2
splitA = all_w[perm[:half]]
splitB = all_w[perm[half:2 * half]]

n_null = min(len(splitA), len(splitB))
se_null = bootstrap_se(all_w, n_null, seed=999)
null_raw = np.array([stat_vec(splitA), stat_vec(splitB), stat_vec(all_w)])
null_sections = torch.tensor(null_raw / se_null, dtype=torch.float64)

null_disagree = cocycle_disagreement(null_sections, overlaps)
null_max = float(null_disagree.max())
TOL_SHEAF = max(8.0, 1.5 * null_max)

print(f"NULL SHEAF disagreement (SE) : {[round(x, 2) for x in null_disagree.tolist()]}")
print(f"null_max                     : {null_max:.2f} SE")
print(f"adaptive tolerance TOL_SHEAF : {TOL_SHEAF:.2f} SE  (= max(8, 1.5×null_max))")
null_status, _, _ = sheaf_status_and_kappa(null_sections, overlaps, tol=TOL_SHEAF)
print(f"null status at TOL_SHEAF      : {null_status}  (valid by construction)")

NULL SHEAF disagreement (SE) : [2.72, 4.75]
null_max                     : 4.75 SE
adaptive tolerance TOL_SHEAF : 8.00 SE  (= max(8, 1.5×null_max))
null status at TOL_SHEAF      : valid  (valid by construction)


In [12]:
# Cell 4.2 — Sheaf gluing check (SE-normalized, adaptive tolerance)
# Conformance: Curry 2014 cellular sheaf cocycle equality (axioms §4).
status, kappa_residual, obstruction_at = sheaf_status_and_kappa(sections, overlaps, tol=TOL_SHEAF)

print(f"tolerance used  : {TOL_SHEAF:.2f} SE")
print(f"sheaf_status    : {status}")
print(f"kappa_residual  : {kappa_residual}")
print(f"obstruction_at  : {obstruction_at}")
print()
names = {"0_2": "Black ↔ All", "1_2": "White ↔ All"}
print(f"Per-overlap disagreement (SE units, tol={TOL_SHEAF:.2f}):")
for ov_row, d in zip(overlaps.tolist(), cocycle_disagreement(sections, overlaps).tolist()):
    key = f"{ov_row[0]}_{ov_row[1]}"
    flag = "OBSTRUCTED" if d >= TOL_SHEAF else "valid"
    print(f"  {names.get(key, key):<14}: {d:.2f}  [{flag}]")

tolerance used  : 8.00 SE
sheaf_status    : valid
kappa_residual  : []
obstruction_at  : None

Per-overlap disagreement (SE units, tol=8.00):
  Black ↔ All   : 5.96  [valid]
  White ↔ All   : 1.26  [valid]


---
## Step 5 — ICR Proxy Using Mortgage and Rent Data

The PUMS household file provides housing cost flows:
- **Owners** (`TEN` ∈ {1,2}): `SMOCP` = Selected Monthly Owner Costs (mortgage + taxes + insurance + HOA + utilities). Preferred over `MRGP` (mortgage payment alone) because it captures total housing-cost interception. Falls back to `MRGP` when `SMOCP` is absent.
- **Renters** (`TEN` ∈ {3,4}): `GRNTP` = Gross Rent (contract rent + utilities paid by landlord if included). Falls back to `RNTP` (contract rent).

**Missing for true ICR:** savings deposits, asset purchases, debt service on non-mortgage debt (student loans, auto, credit cards). These require SCF (`X7575`, `X1055`, installment codes) or CEX (FMLI file). Until that data is joined, `compounding_flow = 0` and ICR = intercepted / income (a partial ratio, not the true compound-to-intercept ratio).

In [13]:
# Cell 5.1 — Intercepted flow (vectorized)
hh_above = hh_above.copy()  # avoid SettingWithCopyWarning
hh_above["intercepted_flow"] = 0.0

# --- Owners ---
owner_mask = hh_above["TEN"].isin([1.0, 2.0])

has_smocp = owner_mask & hh_above["SMOCP"].notna()
hh_above.loc[has_smocp, "intercepted_flow"] = hh_above.loc[has_smocp, "SMOCP"] * 12

has_mrgp_only = owner_mask & hh_above["SMOCP"].isna() & hh_above["MRGP"].notna()
hh_above.loc[has_mrgp_only, "intercepted_flow"] = hh_above.loc[has_mrgp_only, "MRGP"] * 12

# --- Renters ---
renter_mask = hh_above["TEN"].isin([3.0, 4.0])

has_grntp = renter_mask & hh_above["GRNTP"].notna()
hh_above.loc[has_grntp, "intercepted_flow"] = hh_above.loc[has_grntp, "GRNTP"] * 12

has_rntp_only = renter_mask & hh_above["GRNTP"].isna() & hh_above["RNTP"].notna()
hh_above.loc[has_rntp_only, "intercepted_flow"] = hh_above.loc[has_rntp_only, "RNTP"] * 12

hh_above["compounding_flow"] = 0.0  # SCF/CEX data absent
hh_above["icr_partial"] = hh_above["intercepted_flow"] / hh_above["HINCP_adj"].clip(lower=1)

print(f"intercepted_flow > 0 : {(hh_above['intercepted_flow'] > 0).sum():,} households")
print(f"intercepted_flow = 0 : {(hh_above['intercepted_flow'] == 0).sum():,} (TEN=NaN or no cost data)")

intercepted_flow > 0 : 3,392 households
intercepted_flow = 0 : 35 (TEN=NaN or no cost data)


In [14]:
# Cell 5.2 — Median ICR and intercepted flow by race (above nucleation threshold)
icr_summary = (
    hh_above[hh_above["intercepted_flow"] > 0]
    .groupby("hh_race")[["intercepted_flow", "HINCP_adj", "icr_partial"]]
    .median()
)
print("Median ICR and flows (households with non-zero intercepted_flow):")
print(icr_summary.to_string())
print()
print("icr_partial interpretation: housing-cost-intercepted fraction of household income.")
print("A higher icr_partial for Black households = larger share of income absorbed by housing costs.")

Median ICR and flows (households with non-zero intercepted_flow):
         intercepted_flow    HINCP_adj  icr_partial
hh_race                                            
Black             18708.0   88022.1750     0.203772
Other             18840.0  119799.5000     0.157821
White             17046.0  109799.2875     0.146244

icr_partial interpretation: housing-cost-intercepted fraction of household income.
A higher icr_partial for Black households = larger share of income absorbed by housing costs.


### ICR — What Is Missing

The `icr_partial` above captures only the **interception** side (housing cost outflows). A true Intercept-to-Compound Ratio requires:

| Flow | Source | PUMS availability |
|---|---|---|
| Rent / owner costs | `GRNTP`, `SMOCP` | ✓ present (used above) |
| Non-mortgage debt service | Credit card, student loan, auto | ✗ SCF `X7575`, CEX installment codes |
| Savings deposits | Bank / brokerage deposits | ✗ SCF `X3801`–`X3821` |
| Asset purchases | Real estate, equity, retirement contributions | ✗ SCF `X3930`–`X3942`, CEX `TOTXEST` |

**To populate ICR fully:** join SCF public-use extract on race × income-decile strata, or join CEX FMLI quarterly file on the same strata. The `intercepted_flow` column in the DataFrame above is ready to receive the enriched values.

---
## Step 6 — Chile–Barbados Interval Statement

The Chile–Barbados framework compares observed Black household wealth to the cross-national regression prediction at the same income level.  
Reference bands (`CB_LOWER`, `CB_UPPER`): expected median equivalised housing wealth at ~\$79k equivalised income, derived from the OTS 2.0 FAF-W3 cross-national anchors.

**Wealth proxy caveat:** `equiv_wealth` here is housing wealth only (VALP for owners, 0 for renters). Non-housing assets (financial accounts, vehicles, retirement) are absent. All deficit figures below are **lower bounds**.

In [15]:
# Cell 6.1 — Housing-wealth gap and percentile position (Chile–Barbados reframed)
black_hv = hh_above[hh_above["hh_race"] == "Black"]["home_value_per_adult"]
white_hv = hh_above[hh_above["hh_race"] == "White"]["home_value_per_adult"]
black_median_all = black_hv.median()
white_median_all = white_hv.median()
black_median_inc = hh_above[hh_above["hh_race"] == "Black"]["equiv_income"].median()

black_own = hh_above[
    (hh_above["hh_race"] == "Black") & hh_above["TEN"].isin([1.0, 2.0])
]["home_value_per_adult"]
white_own = hh_above[
    (hh_above["hh_race"] == "White") & hh_above["TEN"].isin([1.0, 2.0])
]["home_value_per_adult"]

# Percentile rank of each median within the pooled per-adult housing-wealth distribution.
pooled = hh_above["home_value_per_adult"].values
pct_black = float((pooled < black_median_all).mean() * 100)
pct_white = float((pooled < white_median_all).mean() * 100)

# Chile–Barbados band is a NET-WORTH expectation — reference only, NOT differenced here.
CB_LOWER, CB_UPPER = 200_000, 300_000

print("=== Per-adult housing wealth (gross home value / persons; renters at 0) ===")
print(f"  Black median equiv income           : ${black_median_inc:>12,.0f}")
print(f"  Black median home value/adult        : ${black_median_all:>12,.0f}  (pctile {pct_black:.0f} of pooled)")
print(f"  White median home value/adult        : ${white_median_all:>12,.0f}  (pctile {pct_white:.0f} of pooled)")
print(f"  Owner-only Black median              : ${black_own.median():>12,.0f}  (n={len(black_own):,})")
print(f"  Owner-only White median              : ${white_own.median():>12,.0f}  (n={len(white_own):,})")
print()
print("=== Housing-wealth gap (lower bound; NOT net worth) ===")
print(f"  White − Black (all)    : ${white_median_all - black_median_all:>12,.0f}")
print(f"  White − Black (owners) : ${white_own.median() - black_own.median():>12,.0f}")
print()
print("=== Chile–Barbados reference (net-worth expectation, NOT differenced here) ===")
print(f"  CB band at Black income : ${CB_LOWER:,} – ${CB_UPPER:,} (net worth)")
print()
print("NOTE: figures above are GROSS HOUSING value per adult, not net worth. Mortgage debt")
print("is not subtracted and non-housing assets are absent. The CB band is a net-worth")
print("expectation; comparing it to this housing-only proxy would overstate the deficit, so")
print("we report the housing-wealth gap and Black percentile position instead. See caveat.")

=== Per-adult housing wealth (gross home value / persons; renters at 0) ===
  Black median equiv income           : $      61,451
  Black median home value/adult        : $     110,000  (pctile 34 of pooled)
  White median home value/adult        : $     175,000  (pctile 51 of pooled)
  Owner-only Black median              : $     158,333  (n=291)
  Owner-only White median              : $     200,000  (n=2,346)

=== Housing-wealth gap (lower bound; NOT net worth) ===
  White − Black (all)    : $      65,000
  White − Black (owners) : $      41,667

=== Chile–Barbados reference (net-worth expectation, NOT differenced here) ===
  CB band at Black income : $200,000 – $300,000 (net worth)

NOTE: figures above are GROSS HOUSING value per adult, not net worth. Mortgage debt
is not subtracted and non-housing assets are absent. The CB band is a net-worth
expectation; comparing it to this housing-only proxy would overstate the deficit, so
we report the housing-wealth gap and Black percentile p

### Chile–Barbados Interval — Delaware Interpretation

**Cumulant divergence (Step 3):** `‖Δκ_3‖`, `‖Δκ_4‖` measure how far the Black and White per-adult housing-wealth distributions differ in higher-order structure; the null control collapses them, isolating genuine divergence from sampling noise. The standardized admission gate is a noise guard and is expected to stay silent for independently sampled groups.

**Sheaf obstruction (Step 4):** Disagreement is measured in bootstrap-SE units (`tol=8`). The null control reads `valid`; read the printed real status directly. With a small Delaware Black sample (n≈389), the SE is large, so a `valid` real result reflects limited statistical power, not absence of structure — the national notebook, with n≈70k Black households, is the powered test.

**Housing-wealth framing (Step 6):**
- All wealth figures are **gross home value per adult**, a lower-bound housing-only proxy — **not net worth**.
- Reported as a Black–White housing-wealth **gap** and the **percentile position** of each median within the pooled distribution, not as a dollar deficit against the net-worth Chile–Barbados band.
- The CB \$200k–\$300k band is a net-worth expectation shown for reference only.

**Caveats and next steps:**
1. **Housing wealth only.** Non-housing assets and mortgage debt are absent. Figures are gross housing value, a lower bound.
2. **ICR is partial.** `icr_partial` captures housing-cost interception only. Add SCF/CEX debt-service and savings data for the full compounding-to-interception ratio.
3. **Powered replication:** the national notebook applies the identical pipeline to all 50 states, where the Black sample is large enough to clear the SE-normalized sheaf threshold.